In [7]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from typing import Optional
from pydantic import BaseModel, Field
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class PIOExtraction(BaseModel):
    population: Optional[str] = Field(description="The patients or problem. Return null if none.")
    intervention: Optional[str] = Field(description="The main treatment. Return null if none.")
    outcome: Optional[str] = Field(description="The primary results. Return null if none.")

llm = OllamaLLM(model="llama3.1", format="json", temperature=0)
parser = JsonOutputParser(pydantic_object=PIOExtraction)

def extract_text_from_mask(text, mask):
    words = text.split()
    extracted_spans = []
    current_span = []
    
    for word, label in zip(words, mask):
        if label == 1:
            current_span.append(word)
        else:
            if current_span:
                extracted_spans.append(" ".join(current_span))
                current_span = []
                
    if current_span: 
        extracted_spans.append(" ".join(current_span))
        
    if not extracted_spans:
        return "null"
        
    return " ; ".join(extracted_spans)

def evaluate_extraction(original_text, extracted_text, gt_mask, threshold=0.6):
    """Splits by semicolon and checks each extracted item individually."""
    if not extracted_text or str(extracted_text).lower() in ('null', 'none'):
        return "None", None 
        
    items = [item.strip() for item in str(extracted_text).split(';')]
    
    orig_words = original_text.split()
    item_scores = []
    
    for item in items:
        if not item: continue
        
        ext_words = item.split()
        window = len(ext_words)
        best_ratio, best_start = 0, 0
        
        for i in range(len(orig_words) - window + 1):
            window_text = " ".join(orig_words[i:i+window])
            ratio = SequenceMatcher(None, item.lower(), window_text.lower()).ratio()
            if ratio > best_ratio:
                best_ratio, best_start = ratio, i

        if best_ratio < threshold:
            item_scores.append(0.0) # (Hallucination)
        else:
            mask_slice = gt_mask[best_start : best_start + window]
            if any(label == 1 for label in mask_slice):
                item_scores.append(1.0) # (Hit)
            else:
                item_scores.append(0.0) # (Miss)

    if not item_scores:
        return "None", None

    avg_precision = sum(item_scores) / len(item_scores)
    
    if avg_precision == 1.0:
        return "Hit (Relaxed)", 1.0
    elif avg_precision == 0.0:
        return "Hallucinated", 0.0
    else:
        return "Partial Hit", avg_precision

def run_dynamic_experiment(num_shots, train_texts, train_masks, test_texts, test_masks, embedder, train_embeddings):
    print(f"\n" + "="*60)
    print(f"RUNNING {num_shots}-SHOT DYNAMIC RAG EXPERIMENT")
    print("="*60)

    template = """You are a medical researcher. Extract PIO elements in JSON format.
    CRITICAL RULE: Your extractions MUST be exact, continuous substrings copied directly from the abstract. Do not add or change any words.
    
    {format_instructions}
    
    {few_shot_examples}
    
    --- ACTUAL TASK ---
    Abstract: {abstract}
    Output:"""

    pipeline = PromptTemplate(
        template=template,
        input_variables=["abstract", "few_shot_examples"], 
        partial_variables={
            "format_instructions": parser.get_format_instructions()
        }
    ) | llm | parser

    limit = 30 #en(test_texts)
    results = []
    
    for i in tqdm(range(limit), desc=f"Processing Abstracts ({num_shots}-shot Dynamic)"):
        text = test_texts[i]
        
        try:
            test_embedding = embedder.encode([text])
            
            similarities = cosine_similarity(test_embedding, train_embeddings)[0]
            
            top_k_indices = np.argsort(similarities)[-num_shots:][::-1]
            
            dynamic_few_shot_string = ""
            for idx in top_k_indices:
                train_text = train_texts[idx]
                pop_str = extract_text_from_mask(train_text, train_masks['Pop'][idx])
                int_str = extract_text_from_mask(train_text, train_masks['Int'][idx])
                out_str = extract_text_from_mask(train_text, train_masks['Out'][idx])
                
                dynamic_few_shot_string += f"--- EXAMPLE ---\n"
                dynamic_few_shot_string += f"Abstract: {train_text}\n"
                dynamic_few_shot_string += f"Output: {{\\\"population\\\": \\\"{pop_str}\\\", \\\"intervention\\\": \\\"{int_str}\\\", \\\"outcome\\\": \\\"{out_str}\\\"}}\n\n"

            res = pipeline.invoke({
                "abstract": text,
                "few_shot_examples": dynamic_few_shot_string.strip()
            })
            
            row = {"Text": text}
            
            for short_key, full_key in [('Pop', 'population'), ('Int', 'intervention'), ('Out', 'outcome')]:
                extracted = res.get(full_key)
                status, prec = evaluate_extraction(text, extracted, test_masks[short_key][i])
                
                row.update({
                    f"{short_key}_Extracted": extracted,
                    f"{short_key}_Status": status,
                    f"{short_key}_Precision": prec
                })
                
            results.append(row)
                
        except Exception as e:
            print(f"\nError processing sample {i}: {e}") 

    if results:
        df = pd.DataFrame(results)
        filename = f"pio_dynamic_{num_shots}_shot_results_50.csv"
        df.to_csv(filename, index=False)
        
        print(f"\n{num_shots} SHOT DYNAMIC - TRUE RELAXED PRECISION SCORES")
        print("-" * 50)
        for key in ['Pop', 'Int', 'Out']:
            valid_mask = df[f'{key}_Status'].isin(['Hit (Relaxed)', 'Partial Hit', 'Miss'])
            mean_prec = df.loc[valid_mask, f'{key}_Precision'].mean()
            hallucinations = (df[f'{key}_Status'] == 'Hallucinated').sum()
            
            prec_str = f"{mean_prec * 100:.1f}%" if pd.notna(mean_prec) else "N/A"
            print(f"{key:<12} Precision : {prec_str:<6} | Hallucinations: {hallucinations}")
        print("-" * 50)

def main():
    print("Loading dataset...")
    data = np.load('./data/ebm_nlp_2_00/processed/ebm_abstracts_full.npz', allow_pickle=True)
    
    train_texts = data['train_texts']
    train_masks = {'Pop': data['train_p'], 'Int': data['train_i'], 'Out': data['train_o']}
    
    test_texts = data['test_texts']
    test_masks = {'Pop': data['test_p'], 'Int': data['test_i'], 'Out': data['test_o']}

    print("Loading Sentence Transformer model (this will take a moment the first time)...")
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    
    print("Pre-computing embeddings for all training texts (this saves time during the loop)...")
    train_embeddings = embedder.encode(train_texts[:200])
    print("Embeddings ready!")

    # Test 1, 2, and 3 shots dynamically
    shots_to_test = [1, 2, 3]
    
    for num_shots in shots_to_test:
        run_dynamic_experiment(
            num_shots, 
            train_texts, 
            train_masks, 
            test_texts, 
            test_masks, 
            embedder, 
            train_embeddings
        )

In [8]:
main()

Loading dataset...
Loading Sentence Transformer model (this will take a moment the first time)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pre-computing embeddings for all training texts (this saves time during the loop)...
Embeddings ready!

RUNNING 1-SHOT DYNAMIC RAG EXPERIMENT


Processing Abstracts (1-shot Dynamic): 100%|████| 30/30 [11:18<00:00, 22.61s/it]



1 SHOT DYNAMIC - TRUE RELAXED PRECISION SCORES
--------------------------------------------------
Pop          Precision : 86.8%  | Hallucinations: 0
Int          Precision : 82.6%  | Hallucinations: 2
Out          Precision : 80.2%  | Hallucinations: 0
--------------------------------------------------

RUNNING 2-SHOT DYNAMIC RAG EXPERIMENT


Processing Abstracts (2-shot Dynamic): 100%|████| 30/30 [14:44<00:00, 29.49s/it]



2 SHOT DYNAMIC - TRUE RELAXED PRECISION SCORES
--------------------------------------------------
Pop          Precision : 86.1%  | Hallucinations: 0
Int          Precision : 82.4%  | Hallucinations: 0
Out          Precision : 78.8%  | Hallucinations: 1
--------------------------------------------------

RUNNING 3-SHOT DYNAMIC RAG EXPERIMENT


Processing Abstracts (3-shot Dynamic): 100%|████| 30/30 [18:47<00:00, 37.58s/it]


3 SHOT DYNAMIC - TRUE RELAXED PRECISION SCORES
--------------------------------------------------
Pop          Precision : 85.2%  | Hallucinations: 0
Int          Precision : 80.2%  | Hallucinations: 0
Out          Precision : 79.7%  | Hallucinations: 1
--------------------------------------------------


In [10]:
import pandas as pd

def calculate_metrics(file_paths):
    all_summaries = []
    
    for path in file_paths:
        try:
            df = pd.read_csv(path)
            
            name_parts = path.split('_')
            shots = f"{name_parts[2]}-Shot" if len(name_parts) > 2 else path
            
            summary = {"Experiment": shots}
            
            for pio in ['Pop', 'Int', 'Out']:
                strict_accuracy = df[f'{pio}_Precision'].mean()
                
                stats = df[f'{pio}_Status'].value_counts()
                hits = stats.get('Hit (Relaxed)', 0)
                partial = stats.get('Partial Hit', 0)
                misses = stats.get('Miss', 0)
                halluc = stats.get('Hallucinated', 0)
                
                summary[f"{pio}_Acc"] = f"{strict_accuracy*100:.1f}%"
                summary[f"{pio}_Counts (H/P/M/Halluc)"] = f"{hits}/{partial}/{misses}/{halluc}"
                
            all_summaries.append(summary)
        except Exception as e:
            print(f"Error processing {path}: {e}")
            
    return pd.DataFrame(all_summaries)

my_files = [
    "pio_dynamic_1_shot_results_50.csv", 
    "pio_dynamic_2_shot_results_50.csv", 
    "pio_dynamic_3_shot_results_50.csv"
]

report_table = calculate_metrics(my_files)
print(report_table.to_string(index=False))

Experiment Pop_Acc Pop_Counts (H/P/M/Halluc) Int_Acc Int_Counts (H/P/M/Halluc) Out_Acc Out_Counts (H/P/M/Halluc)
    1-Shot   86.8%                  18/9/0/0   77.1%                 15/13/0/2   80.2%                 15/15/0/0
    2-Shot   86.1%                  22/8/0/0   82.4%                 15/14/0/0   76.1%                 15/14/0/1
    3-Shot   85.2%                  21/9/0/0   80.2%                 15/15/0/0   77.1%                 16/13/0/1
